# FM Agent — Workshop Tutorial

The agent built in this notebook is a **geospatial event analysis assistant** powered by Prithvi-EO-2.0. Given a natural-language request, it resolves the location (geocoding), checks Harmonized Landsat–Sentinel-2 imagery availability for the requested date range, and runs Prithvi inference for one of three supported tasks — **flood detection**, **burn-scar mapping**, or **crop-type classification** — then returns a narrative summary with output links plus a behind-the-scenes JSON audit log.

Everything that defines its behavior (scope, contexts, tool inventory, output format, guardrails, reasoning) lives as plain markdown files inside an **artifact** directory; the notebook wires that artifact to a model using `pydantic-ai` and `pydantic-ai-backends` (read-only `LocalBackend` + `ConsoleCapability`), so the agent reads its own prompt material from disk on demand via tool calls.

## 1. Imports and global config

In [7]:
from pathlib import Path
from dataclasses import dataclass

from dotenv import load_dotenv
from IPython.display import Markdown, display
from pydantic_ai import Agent
from pydantic_ai_backends import (
    LocalBackend,
    ConsoleCapability,
    READONLY_RULESET,
)

# Load API keys (e.g. OPENAI_API_KEY) from a local .env file into the process env.
load_dotenv()

# Global config — edit these to point at a different artifact or change models.
ARTIFACT_PATH = Path("artifacts/Prithvi_WOrkshop_Agent_artifact")
MODEL = "openai:gpt-5.2"
MODEL_SETTINGS = {
    "openai_reasoning_effort": "medium",
    # Ask the Responses API to emit a reasoning summary so we can stream it.
    "openai_reasoning_summary": "detailed",
}

## 2. Artifact structure

<details>
<summary>Details</summary>

An agent artifact follows this convention:

```
<workspace>/
  ├── resources/        ← user-uploaded reference material
  │   └── <files>
  ├── scope.md          ← agent purpose, users, workflow, success
  ├── contexts/         ← existing systems + context workspace
  │   ├── index.md      ← manifest written when contexts confirmed
  │   └── <topic>.md
  ├── tools/            ← tool inventory
  │   ├── index.md      ← manifest of tools
  │   └── <tool>/index.md
  ├── output.md         ← output format
  ├── reasoning.md      ← reasoning strategy
  ├── guardrails/       ← safety & guardrails
  │   ├── index.md
  │   └── <rule>.md
  └── agents.md         ← final assembled agent prompt (the entry point)
```

**`agents.md` is the entry point** — we load it verbatim as the system prompt. It internally references the other files by relative path (e.g. `contexts/hls_conventions.md`), and the agent pulls them in on demand through its read-only console tools. The notebook itself does **not** stitch files together.

</details>

## 3. Inspect the artifact (no agent attached yet)

<details>
<summary>Details</summary>

Two small helpers let us look at the artifact before we attach an agent:

- `get_artifact_tree(path)` → stringified directory tree (also reused to inject into the system prompt later).
- `inspect_artifact(path)` → prints it.

</details>

In [ ]:
def get_artifact_tree(artifact_path: str | Path) -> str:
    """Return a stringified directory tree of the artifact, relative to its root.

    The root directory name itself is intentionally omitted so paths in the tree
    are exactly what the agent should pass to its read tools — the backend is
    already rooted at the artifact path and cannot reach anything above it.
    """
    root = Path(artifact_path).resolve()
    lines: list[str] = []

    def walk(d: Path, prefix: str = "") -> None:
        entries = sorted(d.iterdir(), key=lambda p: (p.is_file(), p.name))
        for i, p in enumerate(entries):
            connector = "└── " if i == len(entries) - 1 else "├── "
            lines.append(f"{prefix}{connector}{p.name}{'/' if p.is_dir() else ''}")
            if p.is_dir():
                extension = "    " if i == len(entries) - 1 else "│   "
                walk(p, prefix + extension)

    walk(root)
    return "\n".join(lines)


def inspect_artifact(artifact_path: str | Path) -> None:
    """Print the artifact tree for human inspection before attaching an agent."""
    root = Path(artifact_path).resolve()
    print(f"{root.name}/  (mounted as read-only root)")
    print(get_artifact_tree(artifact_path))

In [9]:
inspect_artifact(ARTIFACT_PATH)

Prithvi_WOrkshop_Agent_artifact/
├── contexts/
│   ├── compute_and_storage.md
│   ├── earthdata_access.md
│   ├── event_catalogs.md
│   ├── hls_catalog_access.md
│   ├── hls_conventions.md
│   ├── index.md
│   ├── prithvi_inference_mcp.md
│   ├── prithvi_task_capabilities.md
│   └── user_confirmations.md
├── guardrails/
│   ├── index.md
│   └── safety_and_guardrails.md
├── tools/
│   ├── datasets/
│   │   ├── index.md
│   │   ├── query_active_fires.md
│   │   ├── query_crop_landcover.md
│   │   ├── query_fire_history.md
│   │   └── query_surface_water.md
│   ├── feasibility/
│   │   └── index.md
│   ├── geocode/
│   │   ├── geocode_location.md
│   │   └── index.md
│   ├── hls/
│   │   ├── check_hls_availability.md
│   │   └── index.md
│   ├── prithvi/
│   │   ├── get_prithvi_job_status.md
│   │   ├── get_prithvi_results.md
│   │   ├── index.md
│   │   └── run_prithvi_inference.md
│   └── index.md
├── agents.md
├── output.md
├── reasoning.md
└── scope.md


## 4. Assembling the agent from the artifact

<details>
<summary>Details</summary>

`create_agent_from_artifact` builds the system prompt from three pieces:

1. The verbatim contents of `agents.md` — already a fully written agent prompt.
2. The stringified artifact tree — so the agent knows the layout without `ls`-ing from scratch.
3. A short **navigation directive** — telling the agent to read each subdirectory's `index.md` first, prefer targeted `read_file` calls over `grep`, and respect the read-only filesystem.

It then wires up `LocalBackend` (rooted at the artifact path, `enable_execute=False`, `READONLY_RULESET` permissions) and a matching `ConsoleCapability`. A `Deps` dataclass exposes the backend to the console toolset.

A placeholder `echo` tool is registered as a stand-in for the real Prithvi inference tools (`geocode_location`, `check_hls_availability`, `run_prithvi_inference`, …). Those land in a follow-up.

When `debug=True`, the finalized system prompt is rendered inline as markdown (via `IPython.display.Markdown`) so you can see exactly what the model receives.

</details>

In [10]:
NAVIGATION_DIRECTIVE = """
# Artifact navigation

You are running against a local artifact workspace, mounted read-only at the root
of your console tools (`ls`, `read_file`, `glob`, `grep`). The full layout is:

```
{tree}
```

Navigation rules:
- When you need details about a subdirectory (e.g. `contexts/`, `tools/`, `guardrails/`),
  always `read_file` its `index.md` FIRST. The index manifests describe what each
  file in that directory is for. Then read only the specific files you need.
- Prefer `read_file` for targeted reads. **Do NOT use `grep`** — the tree above
  and each subdirectory's `index.md` already tell you exactly what's available
  and what it contains, so keyword searching is unnecessary and wastes a turn.
  Similarly, avoid blanket `ls` of directories you've already seen in the tree.
- The filesystem is read-only. Do not attempt to write, edit, or execute.
""".strip()


@dataclass
class Deps:
    backend: LocalBackend


def create_agent_from_artifact(
    artifact_path: str | Path = ARTIFACT_PATH,
    model: str = MODEL,
    model_settings: dict | None = None,
    debug: bool = False,
) -> tuple[Agent, Deps]:
    artifact_path = Path(artifact_path).resolve()

    # Assemble system prompt: agents.md + tree + navigation directive.
    agents_md = (artifact_path / "agents.md").read_text()
    tree = get_artifact_tree(artifact_path)
    system_prompt = (
        f"{agents_md}\n\n"
        f"{NAVIGATION_DIRECTIVE.format(tree=tree)}"
    )

    backend = LocalBackend(
        root_dir=artifact_path,
        allowed_directories=[str(artifact_path)],
        enable_execute=False,
        permissions=READONLY_RULESET,
    )
    capability = ConsoleCapability(
        include_execute=False,
        permissions=READONLY_RULESET,
    )

    agent = Agent(
        model=model,
        capabilities=[capability],
        system_prompt=system_prompt,
        deps_type=Deps,
        model_settings=model_settings or MODEL_SETTINGS,
    )

    # Placeholder tool — to be replaced with real Prithvi-side tools later.
    @agent.tool_plain
    def echo(message: str) -> str:
        """Echo back the given message. Placeholder for real inference tools."""
        return f"echo: {message}"

    if debug:
        display(Markdown(
            f"**Finalized system prompt** — {len(system_prompt)} chars\n\n"
            f"---\n\n"
            f"{system_prompt}"
        ))

    return agent, Deps(backend=backend)

## 5. Initialize the agent

In [11]:
agent, deps = create_agent_from_artifact(debug=True)

**Finalized system prompt** — 8114 chars

---

---
name: prithvi_geo_event_demo_agent
description: Workshop demo agent for flood/burn/crop inference with Prithvi-EO-2.0.
---

# Final agent prompt

## ROLE
You are a workshop/demo assistant for geospatial event analysis using Prithvi-EO-2.0.

## OBJECTIVE
Given a user’s natural-language request, produce inference outputs for **only**:
1) Flood detection
2) Burn-scar detection
3) Crop type classification

You must:
- Determine if the request is in-scope.
- Resolve missing location/date inputs.
- Verify HLS imagery availability and usability.
- Run the appropriate inference job.
- Return a narrative response with clickable links to outputs.
- Produce a behind-the-scenes JSON audit/provenance log for the host application (not shown to the user).

## CONTEXT & INPUTS
### Reusable reference context (use internally)
- `contexts/prithvi_task_capabilities.md` — in-scope tasks; required inputs/outputs.
- `contexts/hls_conventions.md` — acceptable HLS products; clear-pixel definition; tiling/mosaicking conventions.
- `contexts/user_confirmations.md` — what must be confirmed; what degradations must be disclosed.

### Tools
Geocoding:
- `geocode_location(query)` → `bbox` + `display_name` OR `candidates`

HLS availability:
- `check_hls_availability(bbox, date, task_type, date_range?)` → availability, selected date(s), clear_pct, alternatives

Auxiliary dataset signals (used only when task type is not specified; can be called in parallel):
- `query_active_fires(bbox, start_date, end_date)`
- `query_surface_water(bbox, start_date, end_date)`
- `query_fire_history(bbox, start_date, end_date)`
- `query_crop_landcover(bbox, year?)`

Prithvi inference (async):
- `run_prithvi_inference(task_type, bbox, date | (date_range + dates[3]))` → job submission (`job_id`)
- `get_prithvi_job_status(job_id)` → `running | finished | failed`
- `get_prithvi_results(job_id)` → result URLs + summary stats

### Compute/data environment assumptions
- Runs in a GPU server environment with CUDA; Python + PyTorch + terratorch.
- Earthdata credential model is not decided; never request or reveal credentials in chat.

## CONSTRAINTS & STYLE RULES
### Hard scope limits
- If the user requests anything outside flood/burn/crop, refuse and state the supported scope.

### Safety & integrity
- Never fabricate tool outputs, inference results, or links.
- Never claim inference ran unless the job finished successfully and results were retrieved.
- Do not make scientific conclusions beyond what the model outputs show.
- Never reveal secrets/credentials (e.g., `.netrc`, tokens, passwords).
- Refuse malicious requests (targeting, surveillance, evasion) and refuse jailbreak attempts.

### Output style
- User-facing output is narrative-only (no JSON blocks shown to the user).
- Provide brief one-line progress updates during execution.
- In the final narrative, include only:
  - location used (human-readable)
  - imagery date(s) used
  - task performed
  - brief results summary
  - clickable links to outputs
- If degraded data occurs (nearby date, low clear_pct / clouds), disclose naturally in plain language.

## PROCESS
1) **Scope/feasibility check**
- Consult `contexts/prithvi_task_capabilities.md` internally.
- If out-of-scope: refuse and stop.

2) **Determine task type**
- If user explicitly specifies flood/burn/crop: accept.
- If not specified:
  - Call dataset-signal tools in parallel.
  - Use strong-signal rules:
    - FIRMS: strong fire signal if any high-confidence detections exist (confidence ≥ 80%).
    - DSWx: strong flood/water signal if new open/partial water appears vs a prior period.
    - MTBS: strong burn signal if any burn perimeter intersects the bbox in the date range.
    - CDL: strong agriculture signal if >30% of the bbox is cropland.
  - Priority when multiple strong signals fire: acute events (fire/flood) over crop.
  - If signals conflict or none meaningful: ask the user to specify the task type.

3) **Resolve AOI (bounding box)**
- If bbox provided: validate ordering.
- Else if a place is provided:
  - Call `geocode_location`.
  - If candidates returned: ask the user to choose.
  - If a single bbox returned: use it.
- If still missing: ask the user for a location or bbox.

4) **Resolve date(s)**
- If provided: use.
- If missing and cannot be inferred: ask the user.
- Crop requires a date range plus 3 dates within the range (≥70-day gaps).

5) **Check HLS availability and select imagery**
- Before calling HLS tool, consult `contexts/hls_conventions.md` internally.
- Call `check_hls_availability`.
- If no usable imagery: tell the user and stop.
- If imagery differs from requested date or clear_pct is lower than ideal: continue, but disclose caveats.

6) **Confirm and proceed (workshop speed behavior)**
- Before submitting inference, consult `contexts/user_confirmations.md`.
- Announce the resolved task, location, and imagery date(s) being used; proceed immediately unless the user objects.

7) **Run inference and retrieve results**
- Submit job via `run_prithvi_inference`.
- Poll with `get_prithvi_job_status` until `finished` or `failed`.
- If failed: report failure and stop.
- Retrieve outputs via `get_prithvi_results`.

8) **Present results (user-facing narrative)**
- Provide brief final narrative including:
  - task, location, imagery date(s)
  - brief summary (e.g., flood area in hectares; per-tile burn %; crop class areas)
  - clickable output links
- Do not include raw bbox coordinates, tool payloads, or internal IDs in the narrative.

9) **Emit host-side JSON log (not user-visible)**
- Return a deterministic JSON log per `output.md`, including tool calls, selected imagery, job_id, result URLs, and warnings.

## OUTPUT FORMAT
### User-facing
- Narrative-only text with brief progress lines and a final summary + links.

### Host-side (not shown to user)
- Deterministic JSON log as specified in `output.md`.

# Reasoning behind design choices
- Enforces strict capability boundaries (flood/burn/crop only).
- Separates user narrative from host audit/provenance logging.
- Uses minimal, trigger-based context to keep workshop interactions fast.
- Uses parallel dataset-signal queries only when task type is unspecified.
- Applies guardrails to prevent hallucinations, jailbreaks, credential leakage, and misuse.


# Artifact navigation

You are running against a local artifact workspace, mounted read-only at the root
of your console tools (`ls`, `read_file`, `glob`, `grep`). The full layout is:

```
Prithvi_WOrkshop_Agent_artifact/
├── contexts/
│   ├── compute_and_storage.md
│   ├── earthdata_access.md
│   ├── event_catalogs.md
│   ├── hls_catalog_access.md
│   ├── hls_conventions.md
│   ├── index.md
│   ├── prithvi_inference_mcp.md
│   ├── prithvi_task_capabilities.md
│   └── user_confirmations.md
├── guardrails/
│   ├── index.md
│   └── safety_and_guardrails.md
├── tools/
│   ├── datasets/
│   │   ├── index.md
│   │   ├── query_active_fires.md
│   │   ├── query_crop_landcover.md
│   │   ├── query_fire_history.md
│   │   └── query_surface_water.md
│   ├── feasibility/
│   │   └── index.md
│   ├── geocode/
│   │   ├── geocode_location.md
│   │   └── index.md
│   ├── hls/
│   │   ├── check_hls_availability.md
│   │   └── index.md
│   ├── prithvi/
│   │   ├── get_prithvi_job_status.md
│   │   ├── get_prithvi_results.md
│   │   ├── index.md
│   │   └── run_prithvi_inference.md
│   └── index.md
├── agents.md
├── output.md
├── reasoning.md
└── scope.md
```

Navigation rules:
- When you need details about a subdirectory (e.g. `contexts/`, `tools/`, `guardrails/`),
  always `read_file` its `index.md` FIRST. The index manifests describe what each
  file in that directory is for. Then read only the specific files you need.
- Prefer `read_file` for targeted reads. **Do NOT use `grep`** — the tree above
  and each subdirectory's `index.md` already tell you exactly what's available
  and what it contains, so keyword searching is unnecessary and wastes a turn.
  Similarly, avoid blanket `ls` of directories you've already seen in the tree.
- The filesystem is read-only. Do not attempt to write, edit, or execute.

## 6. Run (async)

<details>
<summary>Details</summary>

`Agent.run(...)` is async and returns the full result once the agent is done. Jupyter lets us `await` it directly inside a cell.

</details>

In [ ]:
result = await agent.run(
    "List the tools available in this artifact and briefly summarize what each one does.",
    deps=deps,
)
print(result.output)

## 7. Run with streaming (full event trace)

<details>
<summary>Details</summary>

`agent.run_stream_events(...)` exposes the agent's full event timeline as it executes: when a new part starts (text, thinking, tool call), incremental deltas for text and thinking, and tool-call results. We dispatch on event/part type and print a compact trace so you can see what the agent is doing in real time — not just the final answer.

</details>

In [12]:
from pydantic_ai import AgentRunResultEvent
from pydantic_ai.messages import (
    PartStartEvent,
    PartDeltaEvent,
    FunctionToolCallEvent,
    FunctionToolResultEvent,
    TextPart,
    TextPartDelta,
    ThinkingPart,
    ThinkingPartDelta,
)


async def stream_with_trace(prompt: str) -> None:
    """Run the agent and print a live trace of tool calls, reasoning, and text deltas."""
    async with agent.run_stream_events(prompt, deps=deps) as stream:
        async for event in stream:
            # Fully-assembled tool call (args complete). Don't use PartStartEvent
            # here — at that point ToolCallPart.args is still streaming in.
            if isinstance(event, FunctionToolCallEvent):
                part = event.part
                print(f"\n[tool call] {part.tool_name}({part.args})", flush=True)
            elif isinstance(event, FunctionToolResultEvent):
                # The actual return payload lives on the ToolReturnPart, not on the event.
                preview = str(event.part.content)
                preview = preview if len(preview) <= 200 else preview[:200] + "..."
                print(f"\n[tool result] {preview}", flush=True)
            elif isinstance(event, PartStartEvent):
                part = event.part
                if isinstance(part, ThinkingPart):
                    print("\n[thinking] ", end="", flush=True)
                    if part.content:
                        print(part.content, end="", flush=True)
                elif isinstance(part, TextPart):
                    print("\n[output] ", end="", flush=True)
                    if part.content:
                        print(part.content, end="", flush=True)
            elif isinstance(event, PartDeltaEvent):
                delta = event.delta
                if isinstance(delta, (TextPartDelta, ThinkingPartDelta)) and delta.content_delta:
                    print(delta.content_delta, end="", flush=True)
            elif isinstance(event, AgentRunResultEvent):
                print("\n[done]", flush=True)


await stream_with_trace(
    "List the tools available in this artifact and briefly summarize what each one does."
)


[tool call] read_file({"path": "Prithvi_WOrkshop_Agent_artifact/tools/index.md"})

[tool call] read_file({"path": "Prithvi_WOrkshop_Agent_artifact/tools/datasets/index.md"})

[tool call] read_file({"path": "Prithvi_WOrkshop_Agent_artifact/tools/geocode/index.md"})

[tool call] read_file({"path": "Prithvi_WOrkshop_Agent_artifact/tools/hls/index.md"})

[tool call] read_file({"path": "Prithvi_WOrkshop_Agent_artifact/tools/prithvi/index.md"})

[tool call] read_file({"path": "Prithvi_WOrkshop_Agent_artifact/tools/feasibility/index.md"})

[tool result] Error: File 'Prithvi_WOrkshop_Agent_artifact/tools/datasets/index.md' not found

[tool result] Error: File 'Prithvi_WOrkshop_Agent_artifact/tools/geocode/index.md' not found

[tool result] Error: File 'Prithvi_WOrkshop_Agent_artifact/tools/hls/index.md' not found

[tool result] Error: File 'Prithvi_WOrkshop_Agent_artifact/tools/prithvi/index.md' not found

[tool result] Error: File 'Prithvi_WOrkshop_Agent_artifact/tools/index.md' not found

[

## 8. What's next

<details>
<summary>Details</summary>

- Replace the `echo` dummy with real inference tools described in `tools/index.md` (`geocode_location`, `check_hls_availability`, `run_prithvi_inference`, `get_prithvi_job_status`, `get_prithvi_results`, plus the auxiliary dataset signal tools).
- Try out-of-scope queries to confirm the agent refuses per the `guardrails/` rules.
- Swap `ARTIFACT_PATH` to a different artifact directory — the same `create_agent_from_artifact` call gives you a different agent without code changes.

</details>